# UC3 — Mobile Phone Screen (Glass): 6. Agentic Flow (End-to-End with Claude Code)

Notebooks 2–5 walked the pipeline one step at a time. This notebook shows how to run
the **entire pipeline from a single prompt** using **Claude Code** and this repo's
`anomalygen` skill. You describe the run; the agent handles fine-tuning (or reusing a
checkpoint), testcase preparation, generation, evaluation, and a per-sample parameter
search — then hands back the results.

> **How commands run in this tutorial.** All pipeline steps run inside the
> `cosmos-predict2` conda environment. In a notebook cell we prefix shell
> commands with `conda run -n cosmos-predict2` (add `--live-stream` to stream
> logs live). If you prefer, open a JupyterLab **Terminal**, run
> `conda activate cosmos-predict2` once, and paste the same commands without
> the `conda run` prefix.
>
> If that environment does not exist yet, build it first with the top-level
> [tutorial/notebooks/0-setup-cuda128.ipynb](../../0-setup-cuda128.ipynb) — see
> the prerequisite note below.

## 6.0 Set the project root

In [ ]:
# Resolve the repository root (the folder containing pyproject.toml) and cd into it,
# so every relative path below (datasets/, checkpoints/, results/, scripts/) resolves.
import os
d = os.getcwd()
while d != "/" and not os.path.exists(os.path.join(d, "pyproject.toml")):
    d = os.path.dirname(d)
LOCAL_PROJECT_DIR = d
os.chdir(LOCAL_PROJECT_DIR)
# Pipeline scripts read the finetuned models & write outputs under the repo root.
os.environ.setdefault("IMAGINAIRE_OUTPUT_ROOT", "./results")
print("Project root:", LOCAL_PROJECT_DIR)

## 6.1 What the agentic flow does

The `anomalygen` skill orchestrates all phases and adds an **agentic search** on top:

| Phase | What happens |
|---|---|
| Setup | Verifies checkpoints are available |
| Fine-tune | Trains adapters on your dataset *(skipped in `mode=inference_only`)* |
| Prep testcase | Places defect masks on clean images (the AMP step) |
| SDG inference | Generates synthetic anomaly images → `original/` |
| Eval | Scores each image (`nn_score`) → `per_sample.csv` |
| SDG refine | Re-runs generation with per-sample `(guidance, crop_ratio)` chosen by the agent from the eval results; repeats `num_search_run` times |
| Assemble | Keeps the best-seen image per sample across rounds → `searched/` |

You get two output buckets: `original/` (first pass) and `searched/` (best-of-search).
Use `searched/reconstructed_image/` as your final synthetic dataset — by construction it
never scores worse than `original/`.

## 6.2 Launch Claude Code

Claude Code is interactive, so run it from a **JupyterLab terminal** (File → New →
Terminal), not a notebook cell:

```bash
conda activate cosmos-predict2
cd "$(git rev-parse --show-toplevel 2>/dev/null || pwd)"   # repo root
claude
```

Type `/` and confirm the **`anomalygen`** skill appears in the list.

## 6.3 The prompt — reuse the released checkpoint (fast)

The quickest end-to-end run skips training and reuses the UC3 checkpoint from
notebook 0. Paste this into Claude Code:

```
Use anomalygen skill with
  mode=inference_only
  name=UC3_phone_agentic_exp
  dataset_dir=datasets/UC3_phone
  defect_spec=datasets/UC3_phone/defect_spec.jsonl
  checkpoint_dir=checkpoints/nvidia/Cosmos-AnomalyGen-Glass-2B
  step=9000
  num_SDG=6
  num_search_run=3
```

### Parameter reference

| Parameter | Meaning |
|---|---|
| `mode` | `inference_only` reuses a checkpoint; `full` trains first; `finetune_only` trains only. |
| `name` | Run label — outputs go to `results/UC3_phone_agentic_exp/`. |
| `dataset_dir` / `defect_spec` | Prepared Phone Screen dataset and its defect spec (from notebook 1). |
| `checkpoint_dir` / `step` | The released checkpoint and its iteration (both required for `inference_only`). |
| `num_SDG` | Total synthetic images to produce (spread across defect types). |
| `num_search_run` | Per-sample refinement rounds (`0` disables search). |

## 6.4 The prompt — full training (production quality)

To reproduce the checkpoint from scratch (several hours on one GPU), use `mode=full`
and drop `checkpoint_dir`/`step`:

```
Use anomalygen skill with
  mode=full
  name=UC3_phone_full_exp
  dataset_dir=datasets/UC3_phone
  defect_spec=datasets/UC3_phone/defect_spec.jsonl
  max_iter=75000
  validation_iter=2000
  save_iter=2000
  num_SDG=9
  num_search_run=5
```

<font color="red">**Note.** `max_iter=75000` is the full-quality setting and takes hours. For a quick
smoke test use a small `max_iter` (e.g. 200) — quality will be low but the flow runs
end-to-end in minutes.</font>

## 6.5 Monitor & view results

Claude reports progress per phase. Training logs (for `mode=full`) also stream to
`results/anomaly_gen/<group>/<name>/.../stdout.log` — `tail -f` it from a second
terminal. After the run completes, inspect the output tree:

In [ ]:
import os
results_dir = os.path.join(LOCAL_PROJECT_DIR, "results/UC3_phone_agentic_exp")
if os.path.exists(results_dir):
    for root, dirs, files in os.walk(results_dir):
        level = root.replace(results_dir, "").count(os.sep)
        print("  " * level + os.path.basename(root) + "/")
        if level < 2:
            for f in files[:3]:
                print("  " * (level + 1) + f)
            if len(files) > 3:
                print("  " * (level + 1) + f"... ({len(files)} files total)")
else:
    print("Run the agentic prompt in a Claude Code terminal first — expected at:", results_dir)

In [ ]:
import glob, os
from IPython.display import Image, display
searched = os.path.join(LOCAL_PROJECT_DIR, "results/UC3_phone_agentic_exp/searched/reconstructed_image")
imgs = sorted(glob.glob(os.path.join(searched, "*.png")))[:5]
if imgs:
    for p in imgs:
        print(os.path.basename(p)); display(Image(p, width=320))
else:
    print("searched/ output not found yet — run the agentic flow first.")

## 6.6 Hosted equivalent — Physical AI Data Factory

NVIDIA's **Physical AI Data Factory** packages this same defect-image-generation
workflow as a launchable driven by natural-language prompts (a "Day 0/Day 1" flow).
For reference, the equivalent prompts there are:

- **UC3** — `"Run day 1 defect image generation workflow for glass using the packaged sample assets. Generate oil, scratch, and stain defects"`

See the workflow docs:
`https://github.com/NVIDIA/physical-ai-data-factory/blob/main/docs/workflows/physical-ai-defect-image-generation/launchable.md`

## Done

This completes the UC3 tutorial. The agentic `searched/` bucket is your highest-quality
synthetic dataset; feed it into [5-pseudo-labeling](./5-pseudo-labeling.ipynb). Point **all
four** inputs at the same `searched/` run so masks/originals stay paired with the generated
images:

- `--gen_image_dir results/UC3_phone_agentic_exp/searched/reconstructed_image`
- `--ori_image_dir results/UC3_phone_agentic_exp/searched/original_image`
- `--mask_dir      results/UC3_phone_agentic_exp/searched/original_mask`
- `--csv_path      results/UC3_phone_agentic_exp/searched/SDG_result.csv`